In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow.keras import layers, models, losses 
from tensorflow.keras.datasets import mnist 
from tensorflow.keras.metrics import Precision, Recall
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

# fixed seed so the results can be reproduced
SEED = 42
tf.keras.utils.set_random_seed(SEED)  # Python, NumPy and TensorFlow
tf.config.experimental.enable_op_determinism()

In [2]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
# imgs
print(x_train.shape == (60000, 28, 28))
print(x_test.shape == (10000, 28, 28))

# labels 
print(y_train.shape == (60000,))
print(y_test.shape == (10000,))

True
True
True
True


In [3]:
# hold out 6,000 training images for validation (same split in every notebook);
# the test set is used only once, for the final evaluation
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=6000, stratify=y_train, random_state=SEED
)
print(f"train: {x_train.shape[0]:,}  validation: {x_val.shape[0]:,}  test: {x_test.shape[0]:,}")

train: 54,000  validation: 6,000  test: 10,000


In [4]:
model = models.Sequential([
    layers.Input((28,28,1)),
    layers.Rescaling(1./255),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(pool_size=(2)),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(pool_size=(2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

In [5]:
model.compile(optimizer='adam', loss=losses.SparseCategoricalCrossentropy(), metrics=['accuracy'],)
model.fit(x=x_train, y=y_train, batch_size=32, epochs=2, validation_data=(x_val, y_val))

Epoch 1/2
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9365 - loss: 0.2080 - val_accuracy: 0.9793 - val_loss: 0.0706
Epoch 2/2
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9767 - loss: 0.0776 - val_accuracy: 0.9815 - val_loss: 0.0680


In [6]:
model_loss, model_accuracy = model.evaluate(x_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9826 - loss: 0.0524


In [7]:
print(f"Loss: {model_loss}")
print(f"Accuracy: {model_accuracy*100:.2f}")

Loss: 0.05237517133355141
Accuracy: 98.26


In [8]:
predictions = model.predict(x_test)
decisions = np.argmax(predictions, axis=1)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [9]:
print(classification_report(y_test, decisions))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       980
           1       1.00      0.99      0.99      1135
           2       0.97      0.99      0.98      1032
           3       1.00      0.97      0.98      1010
           4       0.99      0.98      0.99       982
           5       0.99      0.98      0.98       892
           6       0.97      0.99      0.98       958
           7       0.98      0.98      0.98      1028
           8       0.96      0.99      0.97       974
           9       0.99      0.96      0.97      1009

    accuracy                           0.98     10000
   macro avg       0.98      0.98      0.98     10000
weighted avg       0.98      0.98      0.98     10000



In [10]:
cm = confusion_matrix(y_test, decisions)
df_cm = pd.DataFrame(cm)
df_cm.index.name = "Real"
df_cm.columns.name = "Predicted"
df_cm

Predicted,0,1,2,3,4,5,6,7,8,9
Real,,,,,,,,,,
0,971,0,0,0,0,0,6,1,2,0
1,0,1120,7,0,0,0,2,2,4,0
2,3,0,1024,0,1,0,0,2,2,0
3,0,0,11,979,0,8,0,4,7,1
4,0,0,1,0,966,0,6,0,4,5
5,2,0,1,2,0,875,8,1,3,0
6,1,2,0,0,1,1,953,0,0,0
7,0,1,14,1,1,0,0,1005,2,4
8,3,0,3,0,0,0,2,1,965,0


In [11]:
model.save('../models/digit-recognizer-without-augmentation.keras')